In [ ]:
import os
import re

import janitor
import pandas as pd
from tqdm.notebook import tqdm

from helpers import extract_hashtags, preprocess_tweet, print_tweet_sample

FP_BOTSCORE = "../build/4-classifyAccounts/out/bot-classification.csv"
FP_TWEETS = "../build/1-downloadTweets/tweets-data/"

## Consolidate tweets

In [ ]:
# Get human/bot classification
classified_human_accounts = (
    pd.read_csv(FP_BOTSCORE).fillna(0).query("bot_score<=.5")["user_id"].to_list()
)
classified_human_accounts[:5]

In [ ]:
tweetsFilesList = [
    "jan2018.csv",
    "feb2018.csv",
    "mar2018.csv",
    "apr2018.csv",
    "may2018.csv",
    "jun2018.csv",
    "jul2018.csv",
    "aug2018.csv",
    "sep2018.csv",
    "oct2018.csv",
    "nov2018.csv",
]

# get all tweets in 2018 (Jan 2018 to 6 Nov 2018)
dfTweets = pd.DataFrame()

for fileName in tqdm(tweetsFilesList):
    _df = pd.read_csv(os.path.join(FP_TWEETS, fileName), index_col=0)
    dfTweets = pd.concat([dfTweets, _df], ignore_index=True)

print(len(dfTweets))
dfTweets.head()

In [ ]:
df_gc_human_tweets = (
    dfTweets.query("author_id in @classified_human_accounts")
    .assign(
        preprocessed_content=lambda df_: df_["content"].apply(preprocess_tweet),
        hashtags=lambda df_: df_["preprocessed_content"].apply(
            lambda s: extract_hashtags(s, ignore_metoo=True)
        ),
        n_hashtags=lambda df_: df_["hashtags"].apply(len) + 1,
        tweet_len=lambda df_: df_["preprocessed_content"].apply(len),
        mentions=lambda df_: df_["preprocessed_content"].apply(
            lambda x: re.findall(r"@\s*(\w+)", x)
        ),
    )
    .reset_index(drop=True)
)
df_gc_human_tweets.to_parquet("gc_human_tweets.parquet", index=False)
df_gc_human_tweets

## Tweets per author

In [ ]:
df_tweets_per_author = (
    df_gc_human_tweets.groupby("author_id").size().reset_index(name="tweet_count")
)

print(f"Total tweets: {len(df_gc_human_tweets):,}")
print(f"Unique authors: {len(df_tweets_per_author):,}")
print(f"Mean tweets per author: {df_tweets_per_author['tweet_count'].mean():.2f}")
print(f"Median tweets per author: {df_tweets_per_author['tweet_count'].median():.0f}")
print("\nPercentiles:")
for p in [50, 75, 90, 95, 99, 99.5, 99.9]:
    val = df_tweets_per_author["tweet_count"].quantile(p / 100)
    print(f"  {p}th percentile: {val:.0f} tweets")

## Browse

### SOTU (Jan 31)

In [ ]:
_n = 20

In [ ]:
print_tweet_sample(
    df_gc_human_tweets, n=_n, date="2018-01-31", label="JAN 31, 2018 (SOTU Day)"
)

### Women's march (Jan 21)

In [ ]:
print_tweet_sample(
    df_gc_human_tweets, n=_n, date="2018-01-21", label="JAN 21, 2018 (Women's March)"
)

### CBF reveal (Sep 16)

In [ ]:
print_tweet_sample(
    df_gc_human_tweets,
    n=_n,
    date="2018-09-17",
    label="SEPT 16, 2018 (Blasey Ford Letter Public)",
)

### Kavanaugh Testimony (Sep 27)

In [ ]:
print_tweet_sample(
    df_gc_human_tweets,
    n=_n,
    date="2018-09-27",
    label="SEPT 27, 2018 (Blasey Ford Testimony)",
)

### General

In [ ]:
print_tweet_sample(df_gc_human_tweets, n=_n)

### Collect all

In [ ]:
samples = {
    "SOTU": ("2018-01-31", "SOTU (Jan 31)"),
    "Womens_March": ("2018-01-21", "Women's March (Jan 21)"),
    "CBF_Reveal": ("2018-09-17", "CBF Reveal (Sep 17)"),
    "Kavanaugh": ("2018-09-27", "Kavanaugh Testimony (Sep 27)"),
    "General": (None, "General (all periods)"),
}

all_coded_tweets = []

for sample_key, (date_filter, sample_name) in samples.items():
    if date_filter:
        sample_df = df_gc_human_tweets[
            df_gc_human_tweets["date"].str.startswith(date_filter)
        ]
    else:
        sample_df = df_gc_human_tweets  # for general random

    sample_tweets = sample_df.sample(n=20, random_state=1)

    for idx, row in sample_tweets.iterrows():
        all_coded_tweets.append(
            {
                "tweet_id": idx,
                "sample": sample_name,
                "date": row["date"],
                "content": row["content"],
                "hashtags": ", ".join(row["hashtags"]) if row["hashtags"] else "",
                "favorites": row["favorites"],
                "retweets": row["retweets"],
            }
        )

# Create DataFrame
df_random_sample = pd.DataFrame(all_coded_tweets)

df_random_sample.to_csv("./output/random_sample.csv", index=False)